# RPS PreTrained YOLO Fine-tuning

## g-drive 마운트

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


## 경로 설정

In [2]:
# 다른 경로에 저장하려면, 아래 경로를 수정하세요
if colab :
  save_dir = '/content/drive/MyDrive/files/save/'
  dataset_zip = '/content/drive/MyDrive/files/RPS_Dataset_YOLO.zip'
  dataset_root = '/content/RPS_Dataset_YOLO'
  project_root = '/content/rps_runs'
else :
  save_dir = '../files/save/'
  dataset_zip = '../files/RPS_Dataset_YOLO.zip'
  dataset_root = './RPS_Dataset_YOLO'
  project_root = './rps_runs'

## RPS 데이터셋 준비

In [3]:
!unzip -q -o {dataset_zip}

## RPS 데이터셋의 Label 만들기

In [4]:
# 클래스 이름 순서 (data.yaml과 반드시 일치해야 함)
CLASSES = ['scissors', 'rock', 'paper']

In [5]:
import xml.etree.ElementTree as ET
import os

def convert(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]
    x = (box[0] + box[2]) / 2.0
    y = (box[1] + box[3]) / 2.0
    w = box[2] - box[0]
    h = box[3] - box[1]
    return (x * dw, y * dh, w * dw, h * dh)

def convert_folder(src_folder, dst_labels_folder):
    """
    src_folder 내의 xml 파일들을 읽어 dst_labels_folder에 txt로 저장
    """
    if not os.path.exists(dst_labels_folder):
        os.makedirs(dst_labels_folder)

    xml_files = [f for f in os.listdir(src_folder) if f.endswith('.xml')]

    for filename in xml_files:
        in_file_path = os.path.join(src_folder, filename)
        out_file_path = os.path.join(dst_labels_folder, filename.replace('.xml', '.txt'))

        tree = ET.parse(in_file_path)
        root = tree.getroot()
        size = root.find('size')
        w = int(size.find('width').text)
        h = int(size.find('height').text)

        with open(out_file_path, 'w') as out_file:
            for obj in root.iter('object'):
                cls = obj.find('name').text
                if cls not in CLASSES:
                    continue
                cls_id = CLASSES.index(cls)

                xmlbox = obj.find('bndbox')
                b = (float(xmlbox.find('xmin').text),
                     float(xmlbox.find('ymin').text),
                     float(xmlbox.find('xmax').text),
                     float(xmlbox.find('ymax').text))

                bb = convert((w, h), b)
                out_file.write(f"{cls_id} {' '.join([f'{a:.6f}' for a in bb])}\n")

    print(f"변환 완료: {src_folder} -> {dst_labels_folder} ({len(xml_files)}개)")

# 1. Train 변환 (images/train 폴더 안의 xml -> labels/train 폴더로)
convert_folder(
    src_folder=os.path.join(dataset_root, "images/train"),
    dst_labels_folder=os.path.join(dataset_root, "labels/train")
)

# 2. Test 변환 (images/test 폴더 안의 xml -> labels/test 폴더로)
convert_folder(
    src_folder=os.path.join(dataset_root, "images/test"),
    dst_labels_folder=os.path.join(dataset_root, "labels/test")
)

변환 완료: /content/RPS_Dataset_YOLO/images/train -> /content/RPS_Dataset_YOLO/labels/train (82개)
변환 완료: /content/RPS_Dataset_YOLO/images/test -> /content/RPS_Dataset_YOLO/labels/test (22개)


## Ultralytics 설치 및 import

In [6]:
!pip -q install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 66.5 MB/s eta 0:00:00


## 학습

In [7]:
DATA = f'{dataset_root}/data.yaml'

!yolo detect train model=yolo11n.pt data={DATA} \
  imgsz=320 epochs=80 batch=32 device=0 workers=4 \
  project={project_root} degrees=15.0 flipud=0.5 \
  fliplr=0.5 mosaic=1.0 name=rps_yolo11n

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/RPS_Dataset_YOLO/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320,

## LiteRT로 변환

In [8]:
BEST = f'{project_root}/rps_yolo11n/weights/best.pt'

!yolo export model={BEST} format=tflite imgsz=320 int8=True data={DATA}

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/content/rps_runs/rps_yolo11n/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 7, 2100) (5.2 MB)
E0000 00:00:1771074690.789012    1359 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771074690.796094    1359 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771074690.813751    1359 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:

## tflite 파일 저장

In [9]:
!cp {project_root}/rps_yolo11n/weights/best_saved_model/*.tflite {save_dir}